# Evo-1: Proprioceptive State Utilization Analysis

## Research Question
**Hypothesis**: Proprioceptive state information is NOT being fully utilized by the state encoder.

## Experimental Design
1. **Pass normal state** → Measure gradient flow to action_head.state_encoder
2. **Pass zeros (ablation)** → Measure gradient flow to action_head.state_encoder
3. **Compare**: If gradients barely change, state is underutilized

## What We'll Measure
- Gradient magnitude at state_encoder (CategorySpecificMLP inside action_head)
- Gradient variance across batches
- Performance drop when state is zeroed
- Gradient sensitivity: ∂Loss/∂state_features

**Model**: Evo-1 (0.77B parameters)  
**Architecture** (VERIFIED from MINT-SJTU/Evo-1):  
- embedder: InternVL3-1B (VL backbone)
- action_head: FlowmatchingActionHead
  - **state_encoder**: CategorySpecificMLP (3-layer MLP: state → embeddings)
  
**Key Metric**: If zero state → minimal gradient change = underutilization

**Critical Correction**: Previous assumption of separate "integration_module" was INCORRECT. State encoding happens via CategorySpecificMLP inside action_head, not a separate integration module.

---

# Part 1: Model Analysis & Hook Diagnostics

## 1. Environment Setup

In [ ]:
# Check environment
import sys
IN_COLAB = 'google.colab' in sys.modules

print("🚀 Running on Google Colab")
!nvidia-smi

🚀 Running on Google Colab
Sat Feb 21 03:32:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## Environment Setup for Evo-1

**OFFICIAL SETUP** (3 separate conda environments):

### Conda Environments Created:
1. **evo1_server** (Python 3.10) - For Evo-1 model inference
   - PyTorch 2.5.1 + CUDA 12.1
   - transformers==4.57.6 (NOT 5.0.0 - causes meta tensor issues)
   - flash-attn (CRITICAL - compiles from source, 10-15 min)
   - All VL model dependencies
   
2. **libero_client** (Python 3.8.13) - For LIBERO benchmark
   - PyTorch 1.11.0 + CUDA 11.3
   - robosuite==1.4.1, mujoco==2.3.7
   - LIBERO-specific dependencies
   
3. **metaworld_client** (Python 3.10) - For MetaWorld benchmark
   - Latest PyTorch
   - metaworld, gymnasium
   - MetaWorld-specific dependencies

### ⚠️ Flash Attention 2 - REQUIRED
Flash Attention is **CRITICAL** for Evo-1, not optional:
- InternVL3-1B (VL backbone) requires Flash Attention for proper performance
- Without it: InternVL3 falls back to standard attention with **significantly degraded accuracy**
- **Official version**: transformers==4.57.6 (NOT 5.0.0 - causes initialization issues)
- Installation: Takes 10-15 minutes (compiles from source in evo1_server environment)
- Runs in background, shows last 5 lines of output

### Why Conda?
- Official Evo-1 setup uses separate environments for each component
- evo1_server needs Python 3.10 + specific transformers version

- libero_client needs Python 3.8.13 (official requirement)- Both require Flash Attention and correct transformers version

- Prevents dependency conflicts between benchmarks- **Meta-World MT50**: 80.6% success rate (SOTA)

- **LIBERO**: 94.8% success rate (SOTA)

### Architecture Notes:### Performance Benchmarks (with proper environment):

- **embedder**: InternVL3Embedder (InternVL3-1B VL backbone)

  - Uses Flash Attention 2 for efficient multi-head attention- **NO separate integration_module** (state encoding is inside action_head)

  - Fallback to standard attention significantly impacts performance  - state_encoder: CategorySpecificMLP (3-layer MLP: state → embeddings)
- **action_head**: FlowmatchingActionHead with internal state_encoder

In [ ]:
# Clone Evo-1 repository
import os
from pathlib import Path

print('📦 Setting up Evo-1 repository...')
print('='*60)

# Colab paths
evo1_repo_path = '/content/Evo-1'

print(f'\nRepository path: {evo1_repo_path}')

# Clone Evo-1
if not Path(f'{evo1_repo_path}/.git').exists():
    print('\n📥 Cloning Evo-1 repository...')
    !git clone https://github.com/MINT-SJTU/Evo-1.git {evo1_repo_path}
    print('✅ Evo-1 cloned')
else:
    print('✅ Evo-1 already exists')

print('='*60)
print('✅ Evo-1 repository ready!')
print('='*60)

📦 Setting up Evo-1 repository...

[1/4] Repository path: /content/Evo-1
✅ Evo-1 already cloned

[2/4] 🔧 Patching InternVL3Embedder...
    File: /content/Evo-1/Evo_1/model/internvl3/internvl3_embedder.py

✅ InternVL3Embedder already patched or no changes needed

[3/4] 📦 Installing Evo-1 dependencies in evo1_env...
✅ Evo-1 dependencies installed

⚠️ Re-locking transformers to 4.46.3...
✅ transformers 4.46.3 re-locked

[4/4] 🔍 Verifying installation...
PyTorch: 2.10.0+cu128

Transformers: 4.46.3

Flash Attention: Installed

✅ Evo-1 repository ready!


In [ ]:
# Install dependencies (3 SEPARATE conda environments - official structure)
import os

print('📦 Installing system dependencies...')
!apt-get update -qq
!apt-get install -y -qq git wget ffmpeg libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa > /dev/null 2>&1

print('📦 Installing Miniconda...')
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda > /dev/null 2>&1
!rm /tmp/miniconda.sh
os.environ['PATH'] = f"/opt/conda/bin:{os.environ['PATH']}"
!conda init bash
!conda config --set always_yes yes

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

print('\n📦 Creating 3 conda environments (official Evo-1 structure)...')
print('='*60)

# Environment 1: Evo-1 server (Python 3.10)
print('\n[1/3] evo1_server (Python 3.10) - for Evo-1 model')
!conda create -n evo1_server python=3.10 -y
!conda run -n evo1_server pip install \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# LOG: 28 Jan 2026 transformers == 5.0.0 (latest currently) causes issue with server initialisation
# specifically, issue with meta tensors and internvit initialisation.
# Use previous version 4.57.6 for functionality

!conda run -n evo1_server pip install \
  'numpy>=1.26.4,<2.0' 'transformers==4.57.6' accelerate diffusers \
  einops timm pillow opencv-python-headless \
  websockets pyyaml huggingface_hub tqdm

print('📦 Installing flash-attn (required, ~10-15 min)...')
!conda run -n evo1_server pip install flash-attn --no-build-isolation 2>&1 | tail -5 || echo '⚠️ Flash-attn failed'
print('✅ evo1_server ready')

# Environment 2: LIBERO client (Python 3.8.13 - OFFICIAL requirement)
print('\n[2/3] libero_client (Python 3.8.13) - for LIBERO benchmark')
!conda create -n libero_client python=3.8.13 -y
!conda run -n libero_client pip install \
  'numpy>=1.20,<2.0' robosuite==1.4.1 mujoco==2.3.7 \
  imageio h5py bddl websockets huggingface_hub
!conda run -n libero_client pip install \
  torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 \
  --extra-index-url https://download.pytorch.org/whl/cu113
print('✅ libero_client ready')

# Environment 3: MetaWorld client (Python 3.10)
print('\n[3/3] metaworld_client (Python 3.10) - for MetaWorld benchmark')
!conda create -n metaworld_client python=3.10 -y
!conda run -n metaworld_client pip install \
  mujoco websockets opencv-python packaging huggingface_hub metaworld gymnasium

print('✅ metaworld_client ready')

# Install ipykernel in evo1_server so notebook can use it
print('\n📦 Configuring notebook to use evo1_server environment...')
!conda run -n evo1_server pip install ipykernel -q
!conda run -n evo1_server python -m ipykernel install --user --name=evo1_server --display-name="Python (evo1_server)"

print('\n' + '='*60)
print('✅ All 3 environments created!')
!conda run -n evo1_server python -c "import sys; print(f'  evo1_server: Python {sys.version.split()[0]}')"
!conda run -n libero_client python -c "import sys; print(f'  libero_client: Python {sys.version.split()[0]}')"
!conda run -n metaworld_client python -c "import sys; print(f'  metaworld_client: Python {sys.version.split()[0]}')"

print('\n⚠️  IMPORTANT: Restart runtime and select "Python (evo1_server)" kernel')
print('   OR set Python path for this notebook session:')
import sys
evo1_python = '/opt/conda/envs/evo1_server/bin/python'
if os.path.exists(evo1_python):
    # Update sys.executable and PATH for current session
    os.environ['PATH'] = f'/opt/conda/envs/evo1_server/bin:{os.environ["PATH"]}'
    print(f'✅ Python path updated to use evo1_server environment')
    print(f'   Python: {evo1_python}')
else:
    print(f'⚠️  evo1_server Python not found at: {evo1_python}')

# Verify flash-attn installation in evo1_server
print('\n🔍 Verifying flash-attn installation...')
result = !conda run -n evo1_server python -c "import flash_attn; print(f'flash-attn version: {flash_attn.__version__}')" 2>&1
if any('flash-attn version' in line for line in result):
    print('✅ Flash Attention installed in evo1_server')
else:
    print('⚠️  Flash Attention not found - may need to wait if still compiling')

print('\n' + '='*60)
print('✅ Setup complete! Subsequent cells will use evo1_server environment')
print('='*60)


## 📝 Quick Reference: Using Conda Environments

**Current notebook setup**: Configured to use `evo1_server` environment automatically

**To run commands in specific environments**:
```python
# evo1_server (for model inference)
!conda run -n evo1_server python -c "import torch; print(torch.__version__)"

# libero_client (for LIBERO benchmarks)  
!conda run -n libero_client python script.py

# metaworld_client (for MetaWorld benchmarks)
!conda run -n metaworld_client python script.py
```

**Verify installations**:
```python
# Check Flash Attention
!conda run -n evo1_server python -c "import flash_attn; print(flash_attn.__version__)"

# Check transformers version (should be 4.57.6, NOT 5.0.0)
!conda run -n evo1_server python -c "import transformers; print(transformers.__version__)"
```

**Note**: Subsequent Python cells in this notebook will automatically use the evo1_server environment since we updated the PATH.

In [ ]:
# Clone EmbodiedLLM repository for hooks
!git clone https://github.com/PrithviTM-glitch/EmbodiedLLM.git
%cd EmbodiedLLM
!git checkout AnalyseMultipleHooks
%cd MultipleHooksStudy
print("✅ Hook repository ready")

Cloning into 'EmbodiedLLM'...
remote: Enumerating objects: 535, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 535 (delta 48), reused 110 (delta 39), pack-reused 412 (from 2)
Receiving objects: 100% (535/535), 141.87 MiB | 40.69 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/EmbodiedLLM
Branch 'AnalyseMultipleHooks' set up to track remote branch 'AnalyseMultipleHooks' from 'origin'.
Switched to a new branch 'AnalyseMultipleHooks'
/content/EmbodiedLLM/MultipleHooksStudy


In [ ]:
# Download Evo-1 checkpoints
from huggingface_hub import snapshot_download
from pathlib import Path

# Setup checkpoint directory structure (Colab)
CHECKPOINTS_DIR = Path('/content/checkpoints/')

CHECKPOINTS = {
    'libero': {'repo': 'MINT-SJTU/Evo1_LIBERO', 'dir': CHECKPOINTS_DIR / 'libero'},
    'metaworld': {'repo': 'MINT-SJTU/Evo1_MetaWorld', 'dir': CHECKPOINTS_DIR / 'metaworld'}
}

print('📥 Downloading Evo-1 checkpoints...')
print('='*60)

for name, cfg in CHECKPOINTS.items():
    cfg['dir'].mkdir(parents=True, exist_ok=True)
    model_file = cfg['dir'] / 'mp_rank_00_model_states.pt'
    
    if model_file.exists() and model_file.stat().st_size > 1_000_000:
        print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB (cached)")
    else:
        print(f"⏳ Downloading {name} from {cfg['repo']}...")
        snapshot_download(
            repo_id=cfg['repo'],
            local_dir=str(cfg['dir']),
            local_dir_use_symlinks=False,
        )
        if model_file.exists():
            print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB (downloaded)")
        else:
            print(f"⚠️  {name}: checkpoint file not found after download")

print('\n' + '='*60)
print('✅ All checkpoints ready')
print(f'   Location: {CHECKPOINTS_DIR}')
print('='*60)

## 2. Run Gradient Flow Analysis

**Quick Start**: Run the cell below to perform complete end-to-end gradient flow analysis.

The analysis script (`run_evo1_gradient_analysis.py`) will:
1. Load Evo-1 model in `evo1_server` conda environment (ensuring correct dependencies)
2. Attach gradient hooks to track gradient flow
3. Run baseline analysis with normal state encoder
4. Run ablation analysis with zeroed state encoder output
5. Compare gradients and generate verdict on state encoder utilization

**Metrics Computed:**

*Baseline Metrics (no ablation):*
- **Baseline Contribution Ratio**: state_encoder_grad / total_model_grad  
  Shows what fraction of total model gradients flow through the state encoder  
  (e.g., 0.05 = 5% of total gradient flow)

*Ablation-Based Metrics (comparing baseline vs ablated):*
- **Gradient Change %**: Absolute difference as percentage of baseline (e.g., 40% change)
- **Retention Ratio**: Fraction of gradient strength remaining after ablation (e.g., 0.60 = 60% retained)
- **Reduction Ratio**: Fraction of gradient strength lost due to ablation (e.g., 0.40 = 40% lost)

**Why two types of ratios?**  
- **Baseline contribution** shows state encoder's share of total gradient flow (no ablation needed)
- **Reduction ratio** shows what happens when state encoder is removed (requires ablation)
- Together they give complete picture: "How much flows through?" + "What happens without it?"

**Interpretation:**
- High baseline contribution (>0.05) + High reduction ratio (>0.3) → State encoder is critical
- Low baseline contribution (<0.02) + Low reduction ratio (<0.1) → Model is vision/language-dominated

**Why use a script?** The script executes in the `evo1_server` conda environment which has:
- Flash Attention (required for InternVL3-1B)
- transformers==4.57.6 (NOT 5.0.0 - critical for meta tensor compatibility)
- PyTorch 2.5.1+cu121

Regular notebook cells run in Colab's default Python (missing these dependencies).

In [ ]:
# Run complete gradient flow analysis using runner script
# This runs in evo1_server conda environment which has all required dependencies
# (Flash Attention, transformers==4.57.6, PyTorch 2.5.1, etc.)

import os

print('🔬 Running Evo-1 Gradient Flow Analysis')
print('='*60)
print('This script will:')
print('  1. Load Evo-1 model (in evo1_server conda environment)')
print('  2. Attach gradient hooks')
print('  3. Run baseline analysis (normal state)')
print('  4. Run ablation analysis (zeroed state encoder)')
print('  5. Compare gradients and generate verdict')
print('='*60)
print()

# Change to EmbodiedLLM repo directory (where scripts are)
os.chdir('/content/EmbodiedLLM/MultipleHooksStudy')

# Run the analysis script in evo1_server environment
# This ensures all dependencies (Flash Attention, transformers==4.57.6) are correct
print('⏳ Running analysis (takes 2-3 minutes)...')
print()

!conda run -n evo1_server python scripts/run_evo1_gradient_analysis.py --checkpoint metaworld

print('\n✅ Analysis complete!')
print('   Results saved to: /content/evo1_gradient_analysis_results.json')
print('   Run next cell to display detailed results')
print()
print('💡 To analyze LIBERO checkpoint instead, run:')
print('   !conda run -n evo1_server python scripts/run_evo1_gradient_analysis.py --checkpoint libero')

## 3. Display Analysis Results

The previous cell saved results to `/content/evo1_gradient_analysis_results.json`.  
Run this cell to display the detailed findings.

In [ ]:
# Display the results from the gradient analysis
import json
import os
from pathlib import Path

results_file = '/content/evo1_gradient_analysis_results.json'

if not os.path.exists(results_file):
    print('❌ Results file not found!')
    print(f'   Expected: {results_file}')
    print('   Please run the previous cell (Cell 10) first')
else:
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print('\n' + '='*60)
    print('EVO-1 GRADIENT FLOW ANALYSIS RESULTS')
    print('='*60)
    
    print(f"\n📊 Model Information:")
    print(f"   Model: {results['model']}")
    print(f"   Checkpoint: {results['checkpoint']}")
    print(f"   Device: {results['device']}")
    print(f"   State Encoder: {results['state_encoder']}")
    
    print(f"\n🔬 Gradient Analysis:")
    print(f"   Ablation Method: {results['ablation_method']}")
    print(f"   \n   BASELINE (Normal State):")
    print(f"   Baseline Gradient Norm:  {results['baseline_grad_norm']:.6f}")
    print(f"   Total Model Grad Norm:   {results['total_model_grad_norm']:.6f}")
    print(f"   Baseline Contribution:   {results['baseline_contribution_ratio']:.6f} ({results['baseline_contribution_ratio']*100:.2f}% of total)")
    print(f"   \n   ABLATION (Zeroed State Encoder):")
    print(f"   Ablated Gradient Norm:   {results['ablated_grad_norm']:.6f}")
    print(f"   Absolute Reduction:      {results['gradient_absolute_reduction']:.6f}")
    print(f"   Gradient Change:         {results['gradient_change_pct']:.1f}%")
    print(f"   Retention Ratio:         {results['gradient_retention_ratio']:.3f} ({results['gradient_retention_ratio']*100:.1f}% retained)")
    print(f"   Reduction Ratio:         {results['gradient_reduction_ratio']:.3f} ({results['gradient_reduction_ratio']*100:.1f}% lost)")
    
    print(f"\n📈 Loss Comparison:")
    print(f"   Baseline Loss:  {results['loss_baseline']:.6f}")
    print(f"   Ablated Loss:   {results['loss_ablated']:.6f}")
    
    print(f"\n🎯 VERDICT: {results['verdict']}")
    print(f"\n{results['explanation']}")
    
    print('\n' + '='*60)
    print('Methodology:')
    print('Measured gradient sensitivity when integration_module')
    print('output was replaced with zeros (output ablation).')
    print('='*60)
    
    # Export for later use
    print(f"\n✅ Full results saved to: {results_file}")

✅ Evo-1 hook framework imported


### Advanced Usage: Script Options

The analysis scripts support different checkpoints and custom output locations:

**Analyze LIBERO checkpoint instead of MetaWorld:**
```python
!conda run -n evo1_server python /content/EmbodiedLLM/MultipleHooksStudy/scripts/run_evo1_gradient_analysis.py --checkpoint libero
```

**Custom output location:**
```python
!conda run -n evo1_server python /content/EmbodiedLLM/MultipleHooksStudy/scripts/run_evo1_gradient_analysis.py --checkpoint metaworld --output /content/my_results.json
```

**Load model only (without analysis):**
```python
!conda run -n evo1_server python /content/EmbodiedLLM/MultipleHooksStudy/scripts/load_evo1_model.py --checkpoint metaworld --output /content/model_info.pkl
```

**Why these scripts?**  
- ✅ Execute in correct conda environment (`evo1_server`)
- ✅ Have all required dependencies (Flash Attention, transformers==4.57.6)
- ✅ Version controlled in git repository
- ✅ Reusable across multiple notebooks
- ✅ No need to modify notebook cells for environment compatibility

## 4. Manual Walkthrough (Optional)

**Note**: The automated analysis (Cell in Section 2) already completes the full gradient flow analysis. The cells below provide a detailed manual walkthrough of each step for educational purposes or custom experimentation.

### Load Evo-1 Model Manually

**Repository**: [MINT-SJTU/Evo-1](https://github.com/MINT-SJTU/Evo-1)  
**Paper**: [arXiv:2512.06951](https://arxiv.org/abs/2512.06951)  
**HuggingFace**: [MINT-SJTU/Evo1_MetaWorld](https://huggingface.co/MINT-SJTU/Evo1_MetaWorld)

### Verified Architecture
Based on actual repository code (Evo_1/scripts/Evo1.py, Evo_1/model/action_head/flow_matching.py):

**Main Components:**
1. `self.embedder` - InternVL3Embedder (InternVL3-1B VL backbone)
   - Processes images + text → fused_tokens
   
2. `self.action_head` - FlowmatchingActionHead (flow matching diffusion)
   - Contains internal `self.state_encoder` - CategorySpecificMLP
   - Processes: (fused_tokens + state) → actions
   
**State Encoding:**
- State is encoded via `action_head.state_encoder` (3-layer MLP)
- State embeddings concatenated with fused_tokens as context for transformer
- **NO separate integration_module** (previous assumption was incorrect)

**⚠️ NOTE**: Cell 14 below (manual loading) is DEPRECATED - use Cell 10 instead (official loading function + conda environment)

In [ ]:
# ⚠️ DEPRECATED: Use Cell 10 instead (official loading function + evo1_server conda env)
# This cell uses manual model instantiation which doesn't handle transformers versions correctly

# Load Evo-1 model following official repository pattern
import torch
import os
import json
import sys
from pathlib import Path

print('⚠️  DEPRECATED: This cell is kept for reference only')
print('   Use Cell 10 instead - it uses official loading + conda environment')
print('   Skipping execution...')
print('='*60)

# Uncomment below to use manual loading (not recommended)
"""

print('🏗️ Loading Evo-1 model...')
print('='*60)

# Determine device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Target device: {device}')

# Colab paths
evo1_repo_path = '/content/Evo-1'
checkpoint_dir = '/content/evo1_checkpoint'
evo1_code_path = f'{evo1_repo_path}/Evo_1'

# Verify repository exists
if not os.path.exists(evo1_repo_path):
    raise FileNotFoundError(f"Evo-1 repository not found at: {evo1_repo_path}\nPlease run the setup cells first!")

print(f'Repository path: {evo1_repo_path}')

# Add Evo-1 to path
if evo1_code_path not in sys.path:
    sys.path.insert(0, evo1_code_path)

# Import Evo-1 model
print('\n[1/4] Importing Evo-1 model class...')
from scripts.Evo1 import EVO1
print('✅ EVO1 imported')

# Download checkpoint from HuggingFace
print(f'\n[2/4] Setting up checkpoint directory: {checkpoint_dir}')
if not os.path.exists(checkpoint_dir):
    print('📥 Downloading Evo-1 MetaWorld checkpoint...')
    print('This may take a few minutes...')
    
    from huggingface_hub import snapshot_download
    checkpoint_dir = snapshot_download(
        repo_id="MINT-SJTU/Evo1_MetaWorld",
        local_dir=checkpoint_dir,
        local_dir_use_symlinks=False
    )
    print(f'✅ Checkpoint downloaded to: {checkpoint_dir}')
else:
    print(f'✅ Using existing checkpoint: {checkpoint_dir}')

# Load model config
print('\n[3/4] Loading model configuration...')
config_path = os.path.join(checkpoint_dir, "config.json")
with open(config_path, "r") as f:
    config = json.load(f)

# Set inference config
config["finetune_vlm"] = False
config["finetune_action_head"] = False
config["num_inference_timesteps"] = 32

# CRITICAL FIX: Load on CPU first to avoid meta tensor issues
config["device"] = "cpu"  # ← Load on CPU first!

print(f'✅ Config loaded:')
print(f'   VLM: {config.get("vlm_name", "Unknown")}')
print(f'   Action head: {config.get("action_head", "Unknown")}')
print(f'   Initial device: CPU (will move to {device} after loading)')

# Create model on CPU
print('\n[4/4] Creating Evo-1 model on CPU...')
print('⏳ Loading InternVL3-1B (this takes ~1-2 minutes)...')
model = EVO1(config).eval()
print('✅ Model created on CPU')

# Load checkpoint weights
print('\n📦 Loading checkpoint weights...')
ckpt_path = os.path.join(checkpoint_dir, "mp_rank_00_model_states.pt")
checkpoint = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(checkpoint["module"], strict=True)
print('✅ Checkpoint weights loaded')

# NOW move to target device
print(f'\n🚀 Moving model to {device}...')
model = model.to(device)
print(f'✅ Model moved to {device}')

print('\n' + '='*60)
print('✅ Evo-1 model loaded successfully!')
print(f'   Repository: MINT-SJTU/Evo-1')
print(f'   Checkpoint: {checkpoint_dir}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')
print(f'   Device: {device}')
print('='*60)
"""

🏗️ Loading Evo-1 model...
Device: cuda

[1/4] Importing Evo-1 model class...
✅ EVO1 imported (using patched InternVL3Embedder)

[2/4] Setting up checkpoint directory: /content/evo1_checkpoint
✅ Using existing checkpoint: /content/evo1_checkpoint

[3/4] Loading model configuration...
✅ Config loaded:
   VLM: OpenGVLab/InternVL3-1B
   Action head: flowmatching

[4/4] Creating and loading Evo-1 model...


🏗️ Loading Evo-1 model...
Device: cuda

[1/4] Importing Evo-1 model class...
✅ EVO1 imported (using patched InternVL3Embedder)

[2/4] Setting up checkpoint directory: /content/evo1_checkpoint
✅ Using existing checkpoint: /content/evo1_checkpoint

[3/4] Loading model configuration...
✅ Config loaded:
   VLM: OpenGVLab/InternVL3-1B
   Action head: flowmatching

[4/4] Creating and loading Evo-1 model...


Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'rope_theta'}
Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'rope_theta'}


🏗️ Loading Evo-1 model...
Device: cuda

[1/4] Importing Evo-1 model class...
✅ EVO1 imported (using patched InternVL3Embedder)

[2/4] Setting up checkpoint directory: /content/evo1_checkpoint
✅ Using existing checkpoint: /content/evo1_checkpoint

[3/4] Loading model configuration...
✅ Config loaded:
   VLM: OpenGVLab/InternVL3-1B
   Action head: flowmatching

[4/4] Creating and loading Evo-1 model...


Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'rope_theta'}
Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'rope_theta'}


RuntimeError: Tensor.item() cannot be called on meta tensors

## 4. Discover Model Structure

In [ ]:
hook_manager = Evo1Hooks(model)
structure = hook_manager.discover_model_structure()

print("\n" + "="*60)
print("Evo-1 Model Structure Discovery")
print("="*60)
print(f"Model Name: {structure['model_name']}")
print(f"Has Proprio Encoder: {structure['has_proprio_encoder']}")
print(f"Proprio Encoder Type: {structure['proprio_encoder_type']}")

print(f"\nComponents Found:")
for component, attr in structure['components'].items():
    print(f"  - {component}: {attr}")

if structure.get('integration_module_found'):
    print("\n🎯 Integration Module FOUND!")
    print("   Key innovation in Evo-1 architecture")
print("="*60)

## 5. Attach Analysis Hooks

In [ ]:
# Attach ONLY gradient hooks (focus on integration_module)
print("Attaching gradient hooks...")

hook_manager.attach_gradient_hooks()
print("✓ Gradient flow hooks attached")
print("  → Monitoring integration_module specifically")

print("\n✅ Ready to analyze state utilization!")

### Run Forward and Backward Pass

In [ ]:
# Prepare sample inputs
inputs = {
    'pixel_values': torch.randn(1, 3, 224, 224).to(device).half(),
    'input_ids': torch.randint(0, 50000, (1, 10)).to(device),
    'robot_state': torch.randn(1, 7).to(device).half()
}

model.train()
outputs = model(**inputs)
loss = outputs.mean()
loss.backward()

print("✅ Forward and backward pass completed")
print(f"Loss: {loss.item():.4f}")

### Capture Baseline Gradients

In [ ]:
# Capture baseline gradient flow
results_baseline = hook_manager.get_results()
gradient_baseline = results_baseline.get('gradient_flow', {})

print("\n" + "="*60)
print("BASELINE: Gradient Flow with Normal State")
print("="*60)

# Extract integration_module gradients
baseline_integration_grad = None
if 'layer_profiles' in gradient_baseline:
    layer_profiles = gradient_baseline['layer_profiles']
    if 'integration_module' in layer_profiles:
        baseline_integration_grad = layer_profiles['integration_module']
        print("\n🎯 Integration Module Gradients (NORMAL STATE):")
        print(f"  Gradient norm: {baseline_integration_grad.get('norm', 0):.6f}")
        print(f"  Gradient mean: {baseline_integration_grad.get('mean_gradient', 0):.6f}")
        print(f"  Gradient variance: {baseline_integration_grad.get('gradient_variance', 0):.6f}")

print("="*60)

### Ablation: Zero Out State Encoder Output

**Better Ablation Strategy**: Instead of zeroing the input, we zero the OUTPUT of `integration_module`. 

This directly tests: "What if the state encoder contributed nothing to the model?"

In [ ]:
# Create hook to zero out integration_module's output
ablation_handle = None

def zero_output_hook(module, input, output):
    """Replace integration_module output with zeros"""
    return torch.zeros_like(output)

# Find and hook the integration_module
for name, module in model.named_modules():
    if 'integration_module' in name:
        ablation_handle = module.register_forward_hook(zero_output_hook)
        print(f"✓ Attached ablation hook to: {name}")
        break

# Run ablation: normal input, but state encoder output is zeroed
print("\nRunning ablation (integration_module OUTPUT = zeros)...")
model.train()
hook_manager.reset()
model.zero_grad()

outputs_ablated = model(**inputs)  # Same inputs, but integration_module outputs zeros
loss_ablated = outputs_ablated.mean()
loss_ablated.backward()

# Remove ablation hook
if ablation_handle:
    ablation_handle.remove()

print(f"✓ Ablation complete - state encoder contribution removed")

# Capture ablation gradients
results_ablated = hook_manager.get_results()
gradient_ablated = results_ablated.get('gradient_flow', {})

### Compare Gradients: Normal vs Ablation

In [ ]:
# Extract integration module gradients from ablation run
ablated_integration_grad = results_ablated.get('integration_module', {})
ablated_norm = ablated_integration_grad.get('gradient_norm', 0.0)

print("Integration Module Gradients (ZERO STATE)")
print(f"Gradient norm: {ablated_norm:.6f}\n")

# Calculate change
baseline_norm = baseline_integration_grad.get('gradient_norm', 0.0)
grad_change_pct = abs(baseline_norm - ablated_norm) / baseline_norm * 100 if baseline_norm > 0 else 0

print(f"{'='*60}")
print(f"COMPARISON")
print(f"{'='*60}")
print(f"Normal State Gradient:   {baseline_norm:.6f}")
print(f"Ablation Gradient:       {ablated_norm:.6f}")
print(f"Change:                  {grad_change_pct:.1f}%")
print(f"{'='*60}")

### Verdict: Is Proprioceptive State Utilized?

In [ ]:
# Verdict thresholds
if grad_change_pct < 10:
    verdict = "❌ UNDERUTILIZED"
    explanation = "When integration_module output is zeroed, gradients barely change. The model doesn't rely on state encoder's contribution."
elif grad_change_pct < 30:
    verdict = "⚠️ PARTIALLY UTILIZED"
    explanation = "Some gradient sensitivity when state encoder is removed, but the dependency is weak."
else:
    verdict = "✅ WELL UTILIZED"
    explanation = "Strong gradient response when state encoder output is ablated. The model meaningfully uses state information."

print(f"\n{'='*60}")
print(f"VERDICT: {verdict}")
print(f"{'='*60}")
print(f"\nGradient Change: {grad_change_pct:.1f}%")
print(f"\n{explanation}")
print(f"\nMethodology: Measured gradient sensitivity when integration_module")
print(f"output was replaced with zeros (output ablation, not input ablation).")
print(f"{'='*60}")

# Export results
results = {
    'model': 'Evo-1 (0.77B)',
    'state_encoder': 'integration_module',
    'ablation_method': 'output_ablation',
    'baseline_grad_norm': float(baseline_norm),
    'ablated_grad_norm': float(ablated_norm),
    'gradient_change_pct': float(grad_change_pct),
    'verdict': verdict
}

import json
with open('evo1_state_utilization_results.json', 'w') as f:
    json.dump(results, f, indent=2)
    
print(f"\n✓ Results exported to: evo1_state_utilization_results.json")